<a href="https://colab.research.google.com/github/JuanSc120/inteligencia-artificial-ll/blob/main/Trabajo_PrediccionDeDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELDA 1: PREPARACIÓN DEL ENTORNO Y LIBRERÍAS
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

# Configuración visual para Google Colab
sns.set_theme(style="whitegrid")
np.random.seed(42)
print("✅ Librerías importadas correctamente. Entorno listo.")

In [ ]:
# ==============================================================================
# CELDA 2: GENERACIÓN DE DATOS SINTÉTICOS MEJORADOS (1000 Muestras)
# ==============================================================================
n_samples = 1000

# Variables independientes
estrato = np.random.choice([2, 3, 4, 5, 6], size=n_samples, p=[0.20, 0.30, 0.25, 0.15, 0.10])
area_m2 = np.random.normal(loc=60 + 18 * (estrato - 2), scale=20, size=n_samples).clip(30, 400)
habitaciones = (area_m2 / 28 + np.random.normal(0, 0.5, n_samples)).astype(int).clip(1, 6)
distancia_centro_km = np.random.uniform(1, 22, size=n_samples)
edad_inmueble_anios = np.random.uniform(0, 40, size=n_samples) # Nueva variable

# Cálculo del precio con depreciación por edad y saltos no lineales por estrato
precio_base = 90
precio_m2 = 2.5 * (estrato ** 1.1) # Relación no lineal
precio_cop = (
    precio_base
    + area_m2 * precio_m2
    + habitaciones * 12
    - distancia_centro_km * 3
    - (edad_inmueble_anios * 1.5) # Depreciación
    + np.random.normal(0, 45, n_samples) # Ruido aleatorio
).clip(70, 1500)

df_inmuebles = pd.DataFrame({
    'Estrato': estrato,
    'Area_m2': np.round(area_m2, 1),
    'Habitaciones': habitaciones,
    'Dist_Centro_Km': np.round(distancia_centro_km, 1),
    'Edad_Anios': np.round(edad_inmueble_anios, 1),
    'Precio_Millones': np.round(precio_cop, 1)
})

# Variable Objetivo Binaria (Clasificación): Viviendas Top (Más de 500 Millones)
umbral_top = 500
df_inmuebles['Es_Top'] = (df_inmuebles['Precio_Millones'] >= umbral_top).astype(int)

display(df_inmuebles.head())
print(f"\nDistribución de clases (Es_Top):\n{df_inmuebles['Es_Top'].value_counts(normalize=True)*100}")

In [ ]:
# ==============================================================================
# CELDA 3: ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ==============================================================================
plt.figure(figsize=(10, 6))
correlacion = df_inmuebles.drop(columns=['Es_Top']).corr()

# Heatmap de correlación
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlación - Mercado Inmobiliario Bogotá", fontsize=14)
plt.show()

# Insight: Vemos qué variables impactan más el Precio_Millones antes de entrenar

In [ ]:
# ==============================================================================
# CELDA 4: ENTRENAMIENTO Y COMPARACIÓN - REGRESIÓN (Precio Continuo)
# ==============================================================================
X = df_inmuebles[['Estrato', 'Area_m2', 'Habitaciones', 'Dist_Centro_Km', 'Edad_Anios']]
y_reg = df_inmuebles['Precio_Millones']

X_train, X_test, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# 1. Modelo Base: Regresión Lineal
lr_model = LinearRegression()
lr_model.fit(X_train, y_train_r)
pred_lr = lr_model.predict(X_test)

# 2. Modelo Avanzado: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train_r)
pred_rf = rf_model.predict(X_test)

# Evaluación
tabla_regresion = pd.DataFrame({
    "Métrica": ["MAE (Millones COP)", "RMSE (Millones COP)", "R² (Varianza explicada)"],
    "Regresión Lineal": [mean_absolute_error(y_test_r, pred_lr), np.sqrt(mean_squared_error(y_test_r, pred_lr)), r2_score(y_test_r, pred_lr)],
    "Random Forest": [mean_absolute_error(y_test_r, pred_rf), np.sqrt(mean_squared_error(y_test_r, pred_rf)), r2_score(y_test_r, pred_rf)]
}).round(3)

print("=== COMPARATIVA DE MODELOS DE REGRESIÓN ===")
display(tabla_regresion)

In [ ]:
# ==============================================================================
# CELDA 5: ENTRENAMIENTO Y COMPARACIÓN - CLASIFICACIÓN (Vivienda Top)
# ==============================================================================
y_clf = df_inmuebles['Es_Top']
_, _, y_train_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, random_state=42)

# 1. Regresión Logística
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train_c)
pred_log = log_model.predict(X_test)

# 2. Random Forest Classifier
rfc_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rfc_model.fit(X_train, y_train_c)
pred_rfc = rfc_model.predict(X_test)

# Evaluación
tabla_clasificacion = pd.DataFrame({
    "Métrica (%)": ["Exactitud (Accuracy)", "Precisión", "Sensibilidad (Recall)", "F1-Score"],
    "Regresión Logística": [accuracy_score(y_test_c, pred_log), precision_score(y_test_c, pred_log), recall_score(y_test_c, pred_log), f1_score(y_test_c, pred_log)],
    "Random Forest Classifier": [accuracy_score(y_test_c, pred_rfc), precision_score(y_test_c, pred_rfc), recall_score(y_test_c, pred_rfc), f1_score(y_test_c, pred_rfc)]
})

# Multiplicar por 100 para formato porcentaje
tabla_clasificacion.iloc[:, 1:] = (tabla_clasificacion.iloc[:, 1:] * 100).round(2)

print("=== COMPARATIVA DE MODELOS DE CLASIFICACIÓN ===")
display(tabla_clasificacion)

In [ ]:
# ==============================================================================
# CELDA 6: GRÁFICOS INTERACTIVOS CON PLOTLY
# ==============================================================================

# Gráfico 1: Dispersión interactiva (Real vs Predicho) del mejor modelo (Random Forest)
df_resultados = pd.DataFrame({
    'Precio_Real': y_test_r,
    'Precio_Predicho_RF': pred_rf,
    'Estrato': X_test['Estrato']
})

fig1 = px.scatter(
    df_resultados, x='Precio_Real', y='Precio_Predicho_RF', color='Estrato',
    hover_data=['Estrato'], opacity=0.8,
    title="Random Forest: Precio Real vs Predicho (Pasa el cursor por los puntos)",
    labels={'Precio_Real': 'Precio Real (Millones COP)', 'Precio_Predicho_RF': 'Predicción RF (Millones COP)'}
)
# Añadir línea ideal
fig1.add_trace(go.Scatter(x=[70, 1500], y=[70, 1500], mode='lines', name='Línea Ideal (Perfecta)', line=dict(color='red', dash='dash')))
fig1.show()

# Gráfico 2: Importancia de las variables (¿Qué define el precio en Bogotá según el modelo?)
importancias = pd.DataFrame({
    'Variable': X.columns,
    'Importancia': rf_model.feature_importances_
}).sort_values(by='Importancia', ascending=True)

fig2 = px.bar(
    importancias, x='Importancia', y='Variable', orientation='h',
    title="¿Qué variables impactan más el precio? (Feature Importance - Random Forest)",
    color='Importancia', color_continuous_scale='Viridis'
)
fig2.show()